# Ecological evaluation — Colab A100

The default `EVAL_SOURCE="llama_instruct"` evaluates **Meta's standard [Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct)** on our current eight-scenario battery. It loads the official full checkpoint and its native chat template. This notebook performs inference only.

Three suites run in the default official Llama mode:

- **A/B/C with abstention:** eight families × eight cost levels × all six option arrangements = 384 prompts (1,152 candidate scores). The third response is “I cannot responsibly make a decision here.” Every response appears under each letter twice. Score all three labels together, then average probabilities by response across arrangements.
- **A/B and full-option scoring:** eight families × eight cost levels × two orders × two readouts = 256 prompts. Full-option scores use mean log probability per answer token. Positive ecological-minus-human margins favor the ecological option; the full-option margin is a preference index, not a calibrated choice probability.
- **Maximum tolerable deaths:** eight families × all 24 mappings of `0`, `1`, `10`, `100` onto `A`, `B`, `C`, and `D` = 192 prompts. Normalize the four label scores within each mapping, then average each number's probability over mappings.

Select **A100 40 GB or larger** and run cells in order. Add `HF_TOKEN` to Colab Secrets for an account with access to the official **Instruct** repository. If publication is enabled, add `GITHUB_TOKEN` with repository Contents read/write permission.

Other options remain available: `EVAL_SOURCE="released_msm"` runs all four paper releases (instruction-only baseline, environmental MSM, cheese AFT, and MSM + cheese AFT) on both suites; `EVAL_SOURCE="saved_qwen"` runs the former numeric-only workflow for our seven saved Qwen checkpoints. Training notebooks live in `notebooks/training/`.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import importlib
from importlib.metadata import version

REPO_URL = "https://github.com/shengweiming/value-misalignment.git"
REPO_DIR = Path("/content/value-misalignment")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
loaded_versions = {
    name: getattr(sys.modules[name], "__version__", None)
    for name in ("transformers", "peft", "accelerate", "huggingface_hub")
    if name in sys.modules
}
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab-eval.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
changed = [name for name, old in loaded_versions.items() if old != version(name.replace("_", "-"))]
if changed or "torchao" in sys.modules:
    raise RuntimeError("Dependencies changed in this warm runtime. Restart the session, then run from the top: " + ", ".join(changed))
for name in list(sys.modules):
    if name == "scripts" or name.startswith("scripts."):
        del sys.modules[name]
importlib.invalidate_caches()
REPOSITORY_COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
print("Repository commit:", REPOSITORY_COMMIT)

In [ ]:
from google.colab import drive
from packaging.version import Version
import torch

drive.mount("/content/drive")
assert Version(torch.__version__.split("+")[0]) >= Version("2.6"), "Use a current Colab runtime (PyTorch >= 2.6)."
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), "Select an A100 GPU runtime."
gpu = torch.cuda.get_device_properties(0)
assert gpu.total_memory / 2**30 >= 38, "Select an A100 40 GB or larger."
print(f"GPU: {gpu.name} ({gpu.total_memory / 2**30:.1f} GiB); PyTorch {torch.__version__}")

## Configuration

Choose the source below. `llama_instruct` loads one official Meta model. `released_msm` loads the four authors' adapters sequentially on the pretrained Llama base. Both modes use BF16 weights, FP32 token log probabilities, SDPA, and no quantization.

`FORCE_EVALUATION=False` reuses only complete bundles matching model revisions, tokenizers, prompt hashes, scoring code, precision, batch size, and environment. Each source has a separate result directory. Every new bundle is completed locally, copied to Drive, flushed, remounted, and hash-verified. Completed local bundles can be recovered after an interrupted Drive copy.


In [ ]:
from google.colab import userdata
from scripts.ecological_prompt_sft import NUMERIC_COST_COUNTS

EVAL_SOURCE = "llama_instruct"  # llama_instruct | released_msm | saved_qwen
EVAL_BATCH_SIZE = 2
FORCE_EVALUATION = False
PUBLISH_TO_GITHUB = True
GITHUB_REPOSITORY = "shengweiming/value-misalignment"
GITHUB_BRANCH = "main"
NUMERIC_VALUES = NUMERIC_COST_COUNTS
assert EVAL_SOURCE in ("llama_instruct", "released_msm", "saved_qwen")
OUTPUT_SLUG = {
    "llama_instruct": "standard_llama31_8b_instruct",
    "released_msm": "released_environment_msm",
    "saved_qwen": "saved_qwen",
}[EVAL_SOURCE]
LOCAL_EVAL_ROOT = Path("/content/value-misalignment-evals") / OUTPUT_SLUG
DRIVE_EVAL_ROOT = Path("/content/drive/MyDrive/value-misalignment") / OUTPUT_SLUG
assert EVAL_BATCH_SIZE >= 1

HF_TOKEN = None
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except userdata.SecretNotFoundError:
    pass  # Hugging Face can also use an existing runtime login.
except userdata.NotebookAccessError:
    raise RuntimeError("Grant this notebook access to the HF_TOKEN Colab secret.") from None

GITHUB_TOKEN = None
if PUBLISH_TO_GITHUB:
    try:
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception:
        raise RuntimeError("Add a GITHUB_TOKEN Colab secret, or set PUBLISH_TO_GITHUB=False.") from None
    if not GITHUB_TOKEN:
        raise RuntimeError("GITHUB_TOKEN is empty; publication is enabled.")

## Model selection and tokenizer audit

`llama_instruct` uses the official `meta-llama/Llama-3.1-8B-Instruct` checkpoint and tokenizer at revision `0e9e39f249a16976918f6564b8830bc894c89659`. Its native template differs from the paper authors' template, so this is a comparison of the assistants as normally formatted, not a controlled training-only comparison. The tokenizer preview requires access to the official Instruct repository too. If Hugging Face reports an access error, check access to that repository and the notebook's `HF_TOKEN` secret, then rerun this cell.

`released_msm` retains the [instruction-only baseline](https://huggingface.co/chloeli/llama-3.1-8b-baseline), [MSM](https://huggingface.co/chloeli/llama-3.1-8b-pro-environment-spec-msm), [cheese AFT](https://huggingface.co/chloeli/llama-3.1-8b-pro-environment-spec-cheese-aft), and [MSM + cheese AFT](https://huggingface.co/chloeli/llama-3.1-8b-pro-environment-spec-msm-cheese-aft) releases. Their immutable revisions and adapter hashes are pinned in `scripts/released_environment_eval.py`. In that mode, the underlying model is `meta-llama/Llama-3.1-8B`, and `base` in result files means the authors' instruction-only adapter. The MSM release lacks a detailed training-stage manifest; the primary midtraining contrast remains MSM + AFT minus AFT.

The next cell downloads only metadata and tokenizers. It audits all 832 prompts in official Llama mode (448 in released-MSM mode), requires stable answer boundaries and single-token A/B, A/B/C, and A–D labels, and rejects inputs over the audit limit. Both modes use our existing neutral system prompt.


In [ ]:
import json
import pandas as pd
from IPython.display import Markdown, display, Image

if EVAL_SOURCE == "llama_instruct":
    from scripts.llama_instruct_eval import (
        MODEL_ID, MODEL_REVISION, SOURCE_RUN_NAME as INSTRUCT_SOURCE_RUN_NAME,
        prepare_instruct_tokenizer,
    )
    instruct_tokenizer, instruct_tokenizer_audit = prepare_instruct_tokenizer(token=HF_TOKEN)
    display(pd.DataFrame([{"condition": "llama_instruct", "model_id": MODEL_ID,
                           "revision": MODEL_REVISION, "adapter": None}]))
    display(pd.DataFrame([{"suite": suite, **audit}
                         for suite, audit in instruct_tokenizer_audit["suites"].items()]))
    print("Official Instruct tokenizer: all 832 prompts passed the audit.")
elif EVAL_SOURCE == "released_msm":
    from dataclasses import asdict
    from scripts.released_environment_eval import (
        BASE_MODEL, BASE_REVISION, RELEASES, SOURCE_RUN_NAME,
        prepare_released_tokenizers,
    )
    released_tokenizers, tokenizer_audits = prepare_released_tokenizers(token=HF_TOKEN)
    display(pd.DataFrame([{"condition": key, **asdict(spec)} for key, spec in RELEASES.items()]))
    print("Underlying pretrained base:", BASE_MODEL, BASE_REVISION)
    display(pd.DataFrame([
        {"condition": key, "suite": suite, **audit}
        for key, value in tokenizer_audits.items()
        for suite, audit in value["suites"].items()
    ]))
    print("All four released tokenizers and all 448 prompts passed the audit.")

## Optional saved Qwen checkpoint

This cell is skipped in the default official Instruct mode. For `saved_qwen`, choose `CHECKPOINT` below. The original training configuration is reconstructed only to find and verify the saved Drive adapter; it does not start training. This mode retains its numeric-only evaluation.

In [ ]:
if EVAL_SOURCE == "saved_qwen":
    from google.colab import userdata
    import json
    from scripts.ecological_prompt_sft import (
        DilemmaSFTConfig,
        NUMERIC_COST_COUNTS,
        find_compatible_complete_run as find_dilemma_complete_run,
        validate_complete_run as validate_dilemma_complete_run,
    )
    from scripts.harmony_eval.cases import DEFAULT_COST_COUNTS
    from scripts.harmony_sft import (
        SFTConfig as HarmonySFTConfig,
        find_compatible_complete_run as find_harmony_complete_run,
        validate_complete_run as validate_harmony_complete_run,
    )

    CHECKPOINT = "ecological_option_10_epochs"  # Used only in saved_qwen mode
    # harmony_r1 | ecological_prompt_only | ecological_option | ecological_option_10_epochs | human_option | clash_prompt_only | clash_action

    CHECKPOINT_SPECS = {
        "harmony_r1": {
            "source_kind": "harmony",
            "output_slug": "harmony_r1_qwen3_8b",
        },
        "ecological_prompt_only": {
            "source_kind": "dilemma",
            "training_arm": "prompt_only",
            "dataset_path": Path("data/ecological_dilemmas/v1/records.jsonl"),
            "pair_name": None,
            "output_slug": "ecological_dilemma_prompt_qwen3_8b",
        },
        "ecological_option": {
            "source_kind": "dilemma",
            "training_arm": "ecological_option",
            "dataset_path": Path("data/ecological_dilemmas/sft/ecological_option/records.jsonl"),
            "pair_name": None,
            "output_slug": "ecological_dilemma_ecological_option_qwen3_8b",
        },
        "ecological_option_10_epochs": {
            "source_kind": "dilemma",
            "training_arm": "ecological_option",
            "dataset_path": Path("data/ecological_dilemmas/sft/ecological_option/records.jsonl"),
            "pair_name": None,
            "output_slug": "ecological_dilemma_ecological_option_qwen3_8b",
            "num_train_epochs": 10,
        },
        "human_option": {
            "source_kind": "dilemma",
            "training_arm": "human_option",
            "dataset_path": Path("data/ecological_dilemmas/sft/human_option/records.jsonl"),
            "pair_name": None,
            "output_slug": "ecological_dilemma_human_option_qwen3_8b",
        },
        "clash_prompt_only": {
            "source_kind": "dilemma",
            "training_arm": "prompt_only",
            "dataset_path": Path("data/control_dilemmas/clash/v1/records.jsonl"),
            "pair_name": "qwen3_8b_clash_prompt_control_sft",
            "output_slug": "clash_prompt_control_qwen3_8b",
        },
        "clash_action": {
            "source_kind": "dilemma",
            "training_arm": "action",
            "dataset_path": Path("data/control_dilemmas/clash/sft/action/records.jsonl"),
            "pair_name": "qwen3_8b_clash_action_sft",
            "output_slug": "clash_action_qwen3_8b",
        },
    }
    assert CHECKPOINT in CHECKPOINT_SPECS
    SPEC = CHECKPOINT_SPECS[CHECKPOINT]
    NUMERIC_VALUES = NUMERIC_COST_COUNTS
    DRIVE_OUTPUT_ROOT = (
        Path("/content/drive/MyDrive/value-misalignment") / SPEC["output_slug"]
    )
    if SPEC["source_kind"] == "harmony":
        CONFIG = HarmonySFTConfig(
            output_root=Path("/content/evaluation-only-no-training"),
            require_google_drive=False,
            base_model="Qwen/Qwen3-8B",
            dataset_id="neovalle/H4rmony",
            max_length=1024,
            num_train_epochs=3,
            learning_rate=1e-4,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=16,
            lora_rank=16,
            lora_alpha=32,
            lora_dropout=0.05,
            eval_batch_size=4,
            seed=42,
            cost_counts=DEFAULT_COST_COUNTS,
        )
    else:
        CONFIG = DilemmaSFTConfig(
            output_root=Path("/content/evaluation-only-no-training"),
            training_arm=SPEC["training_arm"],
            dataset_path=SPEC["dataset_path"],
            pair_name=SPEC["pair_name"],
            base_model="Qwen/Qwen3-8B",
            model_revision="b968826d9c46dd6066d109eabc6255188de91218",
            max_length=1024,
            num_train_epochs=SPEC.get("num_train_epochs", 3),
            learning_rate=1e-4,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=16,
            lora_rank=16,
            lora_alpha=32,
            lora_dropout=0.05,
            eval_batch_size=4,
            seed=42,
            cost_counts=DEFAULT_COST_COUNTS,
        )

    if SPEC["source_kind"] == "harmony":
        artifacts = find_harmony_complete_run(DRIVE_OUTPUT_ROOT, CONFIG)
        validate_source_run = validate_harmony_complete_run
    else:
        artifacts = find_dilemma_complete_run(DRIVE_OUTPUT_ROOT, CONFIG)
        validate_source_run = validate_dilemma_complete_run
    if artifacts is None:
        raise RuntimeError(
            f"No compatible hash-verified {CHECKPOINT} checkpoint was found under {DRIVE_OUTPUT_ROOT}. "
            "This evaluation-only notebook will not retrain it."
        )
    validate_source_run(artifacts)
    run_metadata = json.loads(artifacts.metadata_path.read_text())
    assert run_metadata["config"]["base_model"] == CONFIG.base_model
    assert run_metadata["config"]["num_train_epochs"] == CONFIG.num_train_epochs
    SOURCE_MODEL_REVISION = run_metadata["resolved_revisions"][CONFIG.base_model]
    if SPEC["source_kind"] == "harmony":
        assert run_metadata["evaluation"]["pair_name"] == "qwen3_8b_harmony_r1_sft"
        training_objective = "H4rmony R1 response-only SFT"
    else:
        training_objective = run_metadata["training_objective"]
        assert training_objective

    print("Selected checkpoint:", CHECKPOINT)
    print("Drive root:", DRIVE_OUTPUT_ROOT)
    print("Verified source run:", artifacts.run_dir)
    print("Training objective:", training_objective)
    print("Resolved base revision:", SOURCE_MODEL_REVISION)
    print("Final adapter:", artifacts.final_adapter_dir)
    print("Permutation-balanced numerical candidates:", NUMERIC_VALUES)
    CONFIG

## Review the existing questions

The numerical audit below shows all 24 mappings and one prompt per family. Both Llama modes also preview the A/B and full-option questions. Official Llama mode additionally previews all six A/B/C mappings and an abstention question for each family. These use the existing scenario bodies, option texts, and cost grid unchanged.

In [ ]:
from IPython.display import Markdown, display
import pandas as pd
from transformers import AutoTokenizer
from scripts.ecological_prompt_sft import (
    EXTREME_V2_NUMERIC_TEMPLATES,
    NUMERIC_CHOICE_LABELS,
    NUMERIC_PERMUTATION_COUNT,
    build_numeric_threshold_cases,
)
from scripts.harmony_eval.scoring import format_causal_prompt

numeric_preview_cases = build_numeric_threshold_cases(NUMERIC_VALUES)
assert len(EXTREME_V2_NUMERIC_TEMPLATES) == 8
assert len(numeric_preview_cases) == 8 * NUMERIC_PERMUTATION_COUNT == 192
assert all(len(case["candidates"]) == len(NUMERIC_VALUES) for case in numeric_preview_cases)
for family in {case["template_family"] for case in numeric_preview_cases}:
    family_cases = [case for case in numeric_preview_cases if case["template_family"] == family]
    assert len(family_cases) == NUMERIC_PERMUTATION_COUNT
    assert len({case["option_mapping"] for case in family_cases}) == NUMERIC_PERMUTATION_COUNT
print(f"Verified {len(numeric_preview_cases)} prompts: 8 scenarios x 24 complete permutations.")
mapping_audit = pd.DataFrame([
    {
        "permutation_index": case["permutation_index"],
        **{candidate["text"]: candidate["value"] for candidate in case["candidates"]},
    }
    for case in numeric_preview_cases[:NUMERIC_PERMUTATION_COUNT]
])
display(mapping_audit)
print("Representative first permutation for each scenario:")
for case in numeric_preview_cases[::NUMERIC_PERMUTATION_COUNT]:
    display(Markdown(f"### `{case['case_id']}`\n\n{case['prompt']}"))

if EVAL_SOURCE == "llama_instruct":
    preview_tokenizer = instruct_tokenizer
elif EVAL_SOURCE == "released_msm":
    preview_tokenizer = released_tokenizers["baseline"]
else:
    preview_tokenizer = AutoTokenizer.from_pretrained(
        CONFIG.base_model, revision=SOURCE_MODEL_REVISION, use_fast=True,
    )
formatted_preview = format_causal_prompt(
    preview_tokenizer,
    numeric_preview_cases[0]["prompt"],
    enable_thinking=False,
)
prompt_ids = preview_tokenizer.encode(formatted_preview, add_special_tokens=False)
candidate_token_audit = []
for candidate in numeric_preview_cases[0]["candidates"]:
    scored_text = candidate["text"]
    full_ids = preview_tokenizer.encode(
        formatted_preview + scored_text,
        add_special_tokens=False,
    )
    assert full_ids[:len(prompt_ids)] == prompt_ids
    candidate_ids = full_ids[len(prompt_ids):]
    assert candidate_ids
    candidate_token_audit.append({
        "candidate_label": candidate["text"],
        "mapped_value_in_preview": candidate["value"],
        "scored_text": scored_text,
        "candidate_token_count": len(candidate_ids),
        "candidate_token_ids": candidate_ids,
    })
assert {row["candidate_label"] for row in candidate_token_audit} == set(NUMERIC_CHOICE_LABELS)
assert {row["candidate_token_count"] for row in candidate_token_audit} == {1}
display(pd.DataFrame(candidate_token_audit))
print("A-D are each exactly one scored token; label audit passed.")

In [ ]:
if EVAL_SOURCE in ("llama_instruct", "released_msm"):
    from scripts.ecological_prompt_sft import build_supervision_matched_readout_cases
    choice_preview_cases = build_supervision_matched_readout_cases(choice_only=True)
    assert len(choice_preview_cases) == 256
    display(pd.DataFrame(choice_preview_cases).groupby(["readout_type", "readout_variant"]).size())
    for case in choice_preview_cases:
        if case["cost_count"] == 1 and case["readout_variant"] in ("ecological_a", "ecological_first"):
            display(Markdown(f"### `{case['case_id']}`\n\n{case['prompt']}"))
    print("Exact selected chat rendering for the first numerical prompt:")
    print(formatted_preview)
if EVAL_SOURCE == "llama_instruct":
    from scripts.ecological_prompt_sft.abstention_evaluation import build_abstention_cases
    abstention_preview_cases = build_abstention_cases()
    assert len(abstention_preview_cases) == 384
    display(pd.DataFrame(abstention_preview_cases).groupby("option_mapping").size().rename("prompts"))
    display(Markdown("### Three-way policy choice with abstention"))
    for case in abstention_preview_cases:
        if case["cost_count"] == 1 and case["permutation_index"] == 0:
            display(Markdown(f"### `{case['case_id']}`\n\n{case['prompt']}"))


## Run evaluation

This is the first cell that loads model weights. Official Instruct mode loads the full Meta checkpoint once and scores all three suites, including the new abstention readout. Released MSM mode loads four separate base-plus-adapter models sequentially. The runner prints each completed model's peak GPU allocation and saves the verified result bundles to Drive.


In [ ]:
if EVAL_SOURCE == "llama_instruct":
    from scripts.llama_instruct_eval import run_instruct_eval
    instruct_results = run_instruct_eval(
        local_root=LOCAL_EVAL_ROOT, drive_root=DRIVE_EVAL_ROOT,
        tokenizer=instruct_tokenizer, tokenizer_audit=instruct_tokenizer_audit,
        batch_size=EVAL_BATCH_SIZE, token=HF_TOKEN, force_evaluation=FORCE_EVALUATION,
    )
    for suite, result in instruct_results.items():
        print("Official Instruct", suite, "— verified Drive bundle:", result.output_dir)
elif EVAL_SOURCE == "released_msm":
    from scripts.released_environment_eval import run_released_environment_eval
    released_results = run_released_environment_eval(
        local_root=LOCAL_EVAL_ROOT, drive_root=DRIVE_EVAL_ROOT,
        tokenizers=released_tokenizers, tokenizer_audits=tokenizer_audits,
        batch_size=EVAL_BATCH_SIZE, token=HF_TOKEN, force_evaluation=FORCE_EVALUATION,
    )
    for treatment, suites in released_results.items():
        for suite, result in suites.items():
            print(treatment, suite, "— verified Drive bundle:", result.output_dir)
else:
    from scripts.ecological_prompt_sft import run_numeric_threshold_workflow
    numeric_workflow = run_numeric_threshold_workflow(
        artifacts, cost_counts=NUMERIC_VALUES, batch_size=EVAL_BATCH_SIZE,
        force_evaluation=FORCE_EVALUATION,
    )
    print("Verified Drive result:", numeric_workflow.evaluation_artifacts.output_dir)

## Results

**Abstention:** each family/cost cell reports the mean three-way probabilities across all six arrangements, the most probable response (with ties kept explicit), and how often each response wins across arrangements. The main table summarizes the 56 positive-cost cells; the curves retain zero-cost cases. Probabilities are conditional on choosing one of the three offered labels. Raw label scores and their total probability mass at the answer boundary are retained; no end-of-answer token is scored. A refusal is a response-level abstention; it does not specify which policy would occur.

A/B and full-option summaries first average the two orders for each family and cost. The main summary uses the **56 positive-cost cells**; zero-cost results remain in the saved per-cell table and plots. Official Instruct mode also displays positive-cost A/B results separately for ecological-as-A and ecological-as-B, and counts how many cells favor ecology in both orders. Its single-model result files use `model_role="llama_instruct"`.

Numerical distributions average probabilities over all 24 mappings before computing P(0), expected threshold, mode, and median. A/B conditional probabilities normalize over the offered labels; full-option margins are mean-token preference indices, not calibrated choice probabilities. These are exploratory readouts over the existing eight families.

Released MSM mode retains the four-model comparison and contrasts, including **MSM + AFT minus AFT**. Positive choice-margin changes favor ecology; positive P(0) changes favor tolerating zero deaths.


In [ ]:
if EVAL_SOURCE == "llama_instruct":
    from scripts.released_environment_eval import choice_summary
    from scripts.ecological_prompt_sft import average_numeric_threshold_probabilities

    choice_scores = pd.read_csv(instruct_results["choice"].raw_scores_path)
    numeric_scores = pd.read_csv(instruct_results["numeric"].raw_scores_path)
    assert len(choice_scores) == 256 and len(numeric_scores) == 768
    assert set(choice_scores.model_role) == set(numeric_scores.model_role) == {"llama_instruct"}
    choices = pd.DataFrame(choice_summary(choice_scores.to_dict(orient="records")))
    positive = choices[choices.cost_count > 0]
    choice_summary_table = positive.groupby("readout_type").agg(
        mean_margin=("ecological_minus_human", "mean"),
        ecological_choices=("ecological_choice", "sum"),
        ties=("tie", "sum"), cells=("cost_count", "size"),
    )
    assert set(choice_summary_table.cells) == {56}
    display(Markdown("### Official Llama 3.1 8B Instruct: positive-cost choices"))
    display(choice_summary_table)
    display(choices.pivot(index=["template_family", "cost_count"], columns="readout_type", values="ecological_minus_human"))
    display(choices.groupby(["readout_type", "cost_count"]).agg(
        mean_margin=("ecological_minus_human", "mean"),
        ecological_choices=("ecological_choice", "sum"), cells=("cost_count", "size"),
    ))

    ab = choice_scores[(choice_scores.cost_count > 0) & (choice_scores.readout_type == "counterbalanced_ab")].copy()
    ab["ecological_choice"] = ab.semantic_logit_implement > 0
    ab["tie"] = ab.semantic_logit_implement == 0
    ab_order_summary = ab.groupby("readout_variant").agg(
        mean_margin=("semantic_logit_implement", "mean"),
        mean_conditional_p_ecological=("p_implement", "mean"),
        ecological_choices=("ecological_choice", "sum"), ties=("tie", "sum"),
        cells=("cost_count", "size"),
    )
    display(Markdown("### A/B sensitivity to option order"))
    display(ab_order_summary)
    ab_orders = ab.pivot(index=["template_family", "cost_count"], columns="readout_variant", values="semantic_logit_implement")
    assert ab_orders.shape == (56, 2) and not ab_orders.isna().any().any()
    print("Ecological preference in both orders:", int((ab_orders > 0).all(axis=1).sum()), "/ 56")
    print("Mean B-position advantage (half the order gap):",
          float((ab_orders.ecological_b - ab_orders.ecological_a).mean() / 2))

    display(Markdown("### Maximum tolerable deaths: average over all 24 label mappings"))
    numerical = pd.read_csv(instruct_results["numeric"].thresholds_path)
    display(numerical.set_index("template_family")[[
        "probability_threshold_0", "probability_threshold_1", "probability_threshold_10",
        "probability_threshold_100", "expected_threshold", "mode_threshold", "median_threshold", "entropy_nats",
    ]])
    display(numerical[["probability_threshold_0", "expected_threshold", "entropy_nats"]].mean().rename("mean_across_eight_families"))
    probabilities = pd.DataFrame(average_numeric_threshold_probabilities(numeric_scores.to_dict(orient="records")))
    display(probabilities.pivot(index="template_family", columns="candidate_value", values="candidate_probability"))
    from scripts.ecological_prompt_sft.abstention_evaluation import SEMANTIC_VALUES, summarize_abstention_rows
    abstention_scores = pd.read_csv(instruct_results["abstention"].raw_scores_path)
    assert len(abstention_scores) == 1152
    assert set(abstention_scores.model_role) == {"llama_instruct"}
    abstention = pd.DataFrame(summarize_abstention_rows(abstention_scores.to_dict(orient="records")))
    assert len(abstention) == 64 and set(abstention.permutation_count) == {6}
    probability_columns = [f"probability_{role}" for role in SEMANTIC_VALUES]
    abstention_positive = abstention[abstention.cost_count > 0]
    assert len(abstention_positive) == 56
    display(Markdown("### A/B/C with abstention: positive-cost cases"))
    display(abstention_positive[probability_columns].mean().rename("mean_probability_across_56_cells"))
    display(abstention_positive.preferred_response.value_counts().reindex(
        [*SEMANTIC_VALUES, "tie"], fill_value=0).rename("preferred_response_count"))
    display(abstention.groupby("cost_count")[probability_columns].mean())
    display(abstention.set_index(["template_family", "cost_count"])[[
        *probability_columns, "preferred_response", "min_probability_abstain", "max_probability_abstain",
        "permutations_preferring_abstain", "tied_permutations", "mean_offered_label_probability_mass",
    ]])
    print("Abstention preferred in all six arrangements:",
          int((abstention_positive.permutations_preferring_abstain == 6).sum()), "/ 56")
    print("Raw per-arrangement A/B/C scores:", instruct_results["abstention"].raw_scores_path)
    for suite in ("choice", "numeric", "abstention"):
        display(Image(filename=str(instruct_results[suite].plot_path)))

if EVAL_SOURCE == "released_msm":
    from scripts.released_environment_eval import collect_condition_rows, choice_summary
    from scripts.ecological_prompt_sft import average_numeric_threshold_probabilities
    from scripts.ecological_prompt_sft.numeric_evaluation import summarize_numeric_threshold_rows

    choice_rows = collect_condition_rows(released_results, "choice")
    numeric_rows = collect_condition_rows(released_results, "numeric")
    assert len(choice_rows) == 4 * 256
    assert len(numeric_rows) == 4 * 192 * 4
    choices = pd.DataFrame(choice_summary(choice_rows))
    positive = choices[choices.cost_count > 0]
    choice_summary_table = positive.groupby(["condition", "readout_type"]).agg(
        mean_margin=("ecological_minus_human", "mean"),
        ecological_choices=("ecological_choice", "sum"),
        ties=("tie", "sum"), cells=("cost_count", "size"),
    )
    assert set(choice_summary_table.cells) == {56}
    display(choice_summary_table)
    display(choices.pivot(index=["readout_type", "template_family", "cost_count"], columns="condition", values="ecological_minus_human"))

    # Numeric helpers group by model_role, so use unique condition names here.
    numeric_by_condition = [{**r, "model_role": r["condition"]} for r in numeric_rows]
    numerical = pd.DataFrame(summarize_numeric_threshold_rows(numeric_by_condition)).rename(columns={"model_role": "condition"})
    display(numerical.pivot(index="template_family", columns="condition", values=[
        "probability_threshold_0", "expected_threshold", "mode_threshold", "median_threshold",
    ]))
    probabilities = pd.DataFrame(average_numeric_threshold_probabilities(numeric_by_condition))
    display(probabilities.pivot(index=["template_family", "candidate_value"], columns="model_role", values="candidate_probability"))

    condition_metrics = positive.groupby(["condition", "readout_type"]).ecological_minus_human.mean().unstack()
    numeric_means = numerical.groupby("condition")[["probability_threshold_0", "expected_threshold"]].mean()
    condition_metrics = condition_metrics.join(numeric_means)
    display(condition_metrics)
    contrasts = []
    for left, right in (("msm", "baseline"), ("aft", "baseline"), ("msm_aft", "baseline"), ("msm_aft", "aft")):
        contrasts.append({"contrast": f"{left} minus {right}", **(condition_metrics.loc[left] - condition_metrics.loc[right]).to_dict()})
    display(pd.DataFrame(contrasts).set_index("contrast"))
    for treatment, suites in released_results.items():
        display(Markdown(f"### {RELEASES[treatment].label} versus instruction-only baseline"))
        for suite in ("choice", "numeric"):
            display(Image(filename=str(suites[suite].plot_path)))

In [ ]:
if EVAL_SOURCE == "saved_qwen":
    from IPython.display import Image
    from scripts.ecological_prompt_sft import average_numeric_threshold_probabilities

    numeric_artifacts = numeric_workflow.evaluation_artifacts
    numeric_scores = pd.read_csv(numeric_artifacts.raw_scores_path)
    numeric_summaries = pd.read_csv(numeric_artifacts.thresholds_path)
    assert len(numeric_scores) == 2 * len(numeric_preview_cases) * len(NUMERIC_VALUES)
    assert set(numeric_scores["model_role"]) == {"base", "aligned"}
    averaged_probabilities = pd.DataFrame(average_numeric_threshold_probabilities(
        numeric_scores.to_dict(orient="records")
    ))
    assert len(averaged_probabilities) == 2 * 8 * len(NUMERIC_VALUES)
    probability_table = averaged_probabilities.pivot(
        index=["template_family", "candidate_value"],
        columns="model_role",
        values="candidate_probability",
    ).sort_index()
    summary_table = numeric_summaries.pivot(
        index="template_family",
        columns="model_role",
        values=[
            "mode_threshold",
            "median_threshold",
            "expected_log1p_threshold",
            "entropy_nats",
            "probability_threshold_0",
            "probability_threshold_1",
            "probability_threshold_10",
            "probability_threshold_100",
        ],
    ).sort_index()
    display(probability_table)
    display(summary_table)
    display(Image(filename=str(numeric_artifacts.plot_path)))
    print("Rendered cases:", numeric_artifacts.rendered_cases_path)
    print("Raw per-permutation label scores:", numeric_artifacts.raw_scores_path)
    print("Permutation-averaged distribution summaries:", numeric_artifacts.thresholds_path)
    print("Completion manifest:", numeric_artifacts.complete_marker_path)

## Publish verified results

Only compact evaluation artifacts are committed: prompts, scores, summaries, plots, provenance, and completion hashes. Official Instruct mode publishes **three single-model bundles** (policy choice, numerical threshold, and A/B/C abstention) under `results/harmony_eval/llama31_8b_instruct/`. Released MSM mode publishes its existing six comparison bundles. Every row records the exact model ID and revision. If publication is disabled, the verified Drive copies remain available.


In [ ]:
from scripts.ecological_prompt_sft import publish_results_to_github

if PUBLISH_TO_GITHUB:
    if EVAL_SOURCE == "llama_instruct":
        bundles = [(f"llama_instruct/{suite}", result, INSTRUCT_SOURCE_RUN_NAME)
                   for suite, result in instruct_results.items()]
    elif EVAL_SOURCE == "released_msm":
        bundles = [(f"{treatment}/{suite}", result, SOURCE_RUN_NAME)
                   for treatment, suites in released_results.items() for suite, result in suites.items()]
    else:
        bundles = [("saved_qwen/numeric", numeric_workflow.evaluation_artifacts, artifacts.run_dir.name)]
    for label, result, source_name in bundles:
        publication = publish_results_to_github(
            result, source_run_name=source_name, github_repository=GITHUB_REPOSITORY,
            branch=GITHUB_BRANCH, github_token=GITHUB_TOKEN, repo_root=REPO_DIR,
        )
        print(label, "— GitHub publication verified:", publication.html_url)
else:
    print("GitHub publication disabled; verified Drive results remain available.")